In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [2]:
raw = pd.read_json("catalog.json")
courses = pd.DataFrame(raw["courses"][0])
for i in range(1,len(raw)):
    courses = pd.concat([courses, pd.DataFrame(raw["courses"][i])], ignore_index=True)

In [3]:
coursetype = []
for i in courses["code"]:
    coursetype.append(i[4])

In [4]:
lower_div_index = np.where(np.array(coursetype) == '0')[0]
#upper_div_index = np.where(np.array(coursetype) == '1')[0]
#prof_teach_nurse_index = np.where(np.array(coursetype) == '2')[0]
#prof_index = np.where(np.array(coursetype) == '4')[0]

In [5]:
lower_div_desc = courses.iloc[lower_div_index][["code","title","desc"]]

In [6]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(lower_div_desc["desc"])

In [7]:
def course_recommender(phrase, data, top_n = 3):
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(data["desc"])
    phrase_vector = tfidf.transform([phrase])
    sim_scores = cosine_similarity(phrase_vector, tfidf_matrix)[0]
    sim_pairs = list(enumerate(sim_scores))
    sorted_pairs = sorted(sim_pairs, key=lambda x: x[1], reverse=True)
    top_matches = sorted_pairs[:top_n]
    recommended_courses = []
    for i, score in top_matches:
        recommended_courses.append(data['code'].iloc[i])
    return recommended_courses

In [9]:
lower_div_desc.to_json('Lower_div_classes.json')